# Phase 1 - Firm Short Reference Workflow

End-to-end walkthrough of the Phase 1 reference implementation for the use case **firm is short on free inventory for delivery**.

Sections:
1. Load CSV data
2. Load YAML configuration
3. Call each CSV-backed tool independently
4. Run the full ADK / local workflow
5. Inspect evidence bundle and diagnosis
6. Read draft commentary
7. Capture human approval
8. Run the eval suite


In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

repo root: /home/user/AgentExample


## 1. Load CSV data

In [2]:
from settlement_agent.infrastructure import csv_loader
import pandas as pd

positions = pd.DataFrame(csv_loader.load_positions())
settlements = pd.DataFrame(csv_loader.load_settlements())
reference = pd.DataFrame(csv_loader.load_reference())
trade_netting = pd.DataFrame(csv_loader.load_trade_netting())
scenarios = pd.DataFrame(csv_loader.load_scenario_manifest())

print('positions:', positions.shape)
print('settlements:', settlements.shape)
print('reference:', reference.shape)
print('trade_netting:', trade_netting.shape)
scenarios.head()

positions: (11, 19)
settlements: (12, 26)
reference: (10, 22)
trade_netting: (13, 21)


,case_id,scenario_name,primary_reason_code,expected_commentary_category,expected_action,should_generate_qma_draft,required_tools
0,UC-01-FIRM-SHORT,Firm Short - Insufficient Free Inventory,FIRM_SHORT_FREE_INVENTORY,Firm Short,Desk to confirm cover or release/realignment; ...,N_PHASE_1,position_tool|settlement_tool|reference_data_t...
1,UC-01-COVERED,Control Case - Delivery Pending but Inventory ...,NO_POSITION_SHORT,No Firm Short Evidence,Flag as insufficient evidence for firm-short c...,N_PHASE_1,position_tool|settlement_tool|reference_data_t...
2,UC-01-ENCUMBERED,Firm Short due to Pledged or Segregated Inventory,FREE_INVENTORY_ENCUMBERED,Firm Short,Request release from pledge/segregation or alt...,N_PHASE_1,position_tool|settlement_tool|reference_data_t...
3,UC-01-INCOMING-COVER,Firm Short with Incoming Receive Expected to C...,SHORT_PENDING_INCOMING_RECEIVE,Matched Short - Fail to Receive,Link incoming receive; monitor settlement befo...,N_PHASE_1,position_tool|settlement_tool|reference_data_t...
4,UC-01-REALIGNMENT,Firm Short requiring Inventory Realignment,REALIGNMENT_REQUIRED,Inventory Realignment Required,Initiate or recommend realignment instruction,N_PHASE_1,position_tool|settlement_tool|reference_data_t...


## 2. Load YAML configuration

In [3]:
from settlement_agent.utils import yaml_loader

use_case = yaml_loader.load_use_case()
agents = yaml_loader.load_agents()
workflow = yaml_loader.load_workflow()
prompts = yaml_loader.load_prompts()
tools_cfg = yaml_loader.load_tools_config()
policy = yaml_loader.load_policy()
evals = yaml_loader.load_eval_suite()

print('use case:', use_case['use_case_id'], '-', use_case['name'])
print('workflow:', workflow['workflow_id'], 'v' + workflow['workflow_version'])
print('agents:', [a['id'] for a in agents['agents']])
print('tools:', [t['name'] for t in tools_cfg['tools']])
print('policy:', policy['policy_id'])
print('evals:', evals['eval_suite_id'], '(', len(evals['cases']), 'cases )')

validation = yaml_loader.validate_all_configs()
assert all(validation.values()), validation
validation

use case: UC-01-FIRM-SHORT - Firm Short on Free Inventory for Delivery
workflow: firm_short_workflow v1.0.0
agents: ['intake_agent', 'evidence_agent', 'diagnosis_agent', 'commentary_agent', 'policy_hitl_agent']
tools: ['position_tool', 'settlement_tool', 'reference_data_tool', 'trade_netting_tool']
policy: phase1_default
evals: phase1_firm_short_suite ( 5 cases )


{'agents/agents.yaml': True,
 'workflows/firm_short_workflow.yaml': True,
 'prompts/prompts.yaml': True,
 'tools/tools.yaml': True,
 'policies/policy.yaml': True,
 'evals/eval_cases.yaml': True}

## 3. Call each CSV-backed tool independently

In [4]:
from settlement_agent.tools.position_tool import call_position_tool
from settlement_agent.tools.settlement_tool import call_settlement_tool
from settlement_agent.tools.reference_data_tool import call_reference_data_tool
from settlement_agent.tools.trade_netting_tool import call_trade_netting_tool

pos = call_position_tool({'account_id': 'ACC-DLV-001', 'security_id': 'SEC-US-0001'})
stl = call_settlement_tool({'instruction_id': 'SI-DLV-1001'})
ref = call_reference_data_tool({'security_id': 'SEC-US-0001'})
trd = call_trade_netting_tool({'account_id': 'ACC-DLV-001', 'security_id': 'SEC-US-0001'})

print('position records :', pos.record_count)
print('settlement records:', stl.record_count)
print('reference records :', ref.record_count)
print('trade_netting recs:', trd.record_count)
pos.records[0]

position records : 1
settlement records: 1
reference records : 1
trade_netting recs: 1


{'position_evidence_id': 'POS-EV-1001',
 'account_id': 'ACC-DLV-001',
 'security_id': 'SEC-US-0001',
 'settlement_location': 'DTC',
 'total_position_qty': 220000.0,
 'free_position_qty': 85000.0,
 'pledged_qty': 90000.0,
 'segregated_qty': 25000.0,
 'on_loan_qty': 20000.0,
 'pending_receive_qty': 0.0,
 'pending_delivery_qty': 150000.0,
 'free_short_qty': 65000.0,
 'inventory_status': 'INSUFFICIENT_FREE_INVENTORY',
 'snapshot_ts': '2026-05-18T14:25:00Z',
 'source_latency_seconds': 18,
 'confidence_score': 0.99}

## 4. Run the full workflow

`run_workflow()` uses Google ADK if installed and otherwise falls back to a local deterministic runner. Either path produces the same `SessionState`.

In [5]:
from settlement_agent.application.workflow import run_workflow, adk_available

print('ADK available?', adk_available())

state = run_workflow('SI-DLV-1001')
print('run_id:', state.run_id)
print('scenario:', state.classification.scenario_label)
print('reason_code:', state.diagnosis.reason_code)
print('is_firm_short:', state.diagnosis.is_firm_short)
print('approval:', state.approval.status, '| final?', state.is_final())

ADK available? False
run_id: run-0a048e28088a
scenario: firm_short
reason_code: FIRM_SHORT_FREE_INVENTORY
is_firm_short: True
approval: pending | final? False


## 5. Inspect the evidence bundle

In [6]:
ev = state.evidence
print('tools called:', ev.tools_called())
print('position rows :', len(ev.position))
print('settlement rows:', len(ev.settlement))
print('reference rows :', len(ev.reference))
print('trade rows     :', len(ev.trade_netting))

import json
print('\nposition[0] =', json.dumps(ev.position[0].model_dump(), indent=2))

tools called: ['settlement_tool', 'position_tool', 'reference_data_tool', 'trade_netting_tool']
position rows : 1
settlement rows: 1
reference rows : 1
trade rows     : 1

position[0] = {
  "position_evidence_id": "POS-EV-1001",
  "account_id": "ACC-DLV-001",
  "security_id": "SEC-US-0001",
  "settlement_location": "DTC",
  "total_position_qty": 220000.0,
  "free_position_qty": 85000.0,
  "pledged_qty": 90000.0,
  "segregated_qty": 25000.0,
  "on_loan_qty": 20000.0,
  "pending_receive_qty": 0.0,
  "pending_delivery_qty": 150000.0,
  "free_short_qty": 65000.0,
  "inventory_status": "INSUFFICIENT_FREE_INVENTORY",
  "snapshot_ts": "2026-05-18T14:25:00Z",
  "source_latency_seconds": 18,
  "confidence_score": 0.99
}


## 6. Draft commentary

In [7]:
print(state.commentary.text)
print('\nevidence_refs:', state.commentary.evidence_refs)
print('policy passed?', state.policy.passed)
print('policy findings:', state.policy.findings)

Delivery remains pending for Alpha Corp Common Stock. Free position is below the pending delivery quantity as of 2026-05-18T14:18:00Z. Incoming receives do not currently cover the shortfall. Desk action required: confirm cover, realignment, or release instruction.

evidence_refs: ['POS-EV-1001', 'SET-EV-2001', 'REF-SEC-001', 'TRD-EV-3001']
policy passed? True
policy findings: []


## 7. Capture human approval

Phase 1 supports `approved`, `rejected`, or `needs_edit`. The workflow does not finalise unless approval is captured. QMA send is **not** triggered automatically by this notebook.

In [8]:
approval = 'approved'  # change to 'rejected' or 'needs_edit' to see other paths
reviewer = 'ops_analyst_demo'

final_state = run_workflow('SI-DLV-1001', approval_status=approval, reviewer=reviewer)
print('approval:', final_state.approval.status)
print('reviewer:', final_state.approval.reviewer)
print('decided_at:', final_state.approval.decided_at)
print('is_final:', final_state.is_final())

approval: approved
reviewer: ops_analyst_demo
decided_at: 2026-05-19 03:00:49.812119
is_final: True


## 8. Run the eval suite

In [9]:
from settlement_agent.application.eval_runner import run_eval_suite

results = run_eval_suite()
for r in results:
    status = 'PASS' if r.passed else 'FAIL'
    print(f'[{status}] {r.scenario_id}')
    if not r.passed:
        print('  checks:', r.checks)
        print('  notes :', r.notes)
passed = sum(1 for r in results if r.passed)
print(f'\n{passed}/{len(results)} eval cases passed')

[PASS] eval_firm_short_confirmed
[PASS] eval_incoming_receive_covers
[PASS] eval_insufficient_evidence
[PASS] eval_counterparty_short
[PASS] eval_netting_mismatch

5/5 eval cases passed
